# Day 10 — Vectorized Operations & Aggregations

This notebook covers Vectorized Math Operations (ufuncs), Axis Aggregations, Array Sorting, and Structured Arrays

## Validation goals
- Understand NumPy universal functions (`ufuncs`) and perform memory-efficient operations using the `out` parameter
- Compute axis-based aggregations (`np.mean`, `np.sum`) across multi-dimensional arrays
- Sort arrays and synchronize parallel data using `np.argsort()`
- Define custom structured array `dtypes` to handle heterogenous data without type upcasting
- Combine structured arrays with vectorized filtering and boolean masking

NumPy operations rely on universal functions (`ufuncs`), which execute element-wise operations directly in compiled C code. When performing reduction or aggregation operations—like finding a total or an average across a 2D matrix—we specify the axis parameter:

`axis=0:` Collapses rows to operate down each column ⬇️

`axis=1`: Collapses columns to operate across each row ➡️

### Guiding Question: 

If you execute `np.mean(readings, axis=0)`, what shape will the resulting array have, and what does each value in that array represent?

Suppose we have a 2D array representing sensor temperature readings across 3 sensors over 2 hours:

`readings = np.array([
    [20.0, 25.0],  # Sensor 1
    [22.0, 24.0],  # Sensor 2
    [19.0, 21.0]   # Sensor 3
])`

In [ ]:
np.mean(readings, axis=0)
# Output: array([20.33333333, 23.33333333])

The resulting array will have a shape of (2,), meaning it contains 2 values

Specifying axis=0 collapses the 3 rows (sensors) down to a 1D array of shape (2,), giving us the average reading across all sensors for Hour 1 (20.33) and Hour 2 (23.33).

### Universal Functions (ufuncs) & In-Place Computation


Under the hood, NumPy functions like `np.mean()`, `np.add()`, or `np.exp()` are vectorized `ufuncs`. `Ufuncs` process array elements in fast C code without Python loop overhead.

To save memory on large datasets, ufuncs accept an optional out parameter to store the result directly in an existing array instead of allocating new memory:

In [7]:
import numpy as np


# In-place addition using the out parameter
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
result = np.empty_like(a)

print(f"{np.add(a, b, out=result)}")

[11. 22. 33.]


### Sorting Arrays


NumPy provides two main ways to sort data:

`np.sort(arr)`: Returns a sorted copy of the array.

`arr.sort()`: Sorts the array in-place.

`np.argsort(arr)`: Returns the indices that would sort the array (invaluable when sorting related datasets simultaneously).

### Structured Arrays


When handling datasets with mixed data types (like `strings`, `integers`, and `floats`), standard NumPy arrays upcast everything to a single type (like `strings`). Structured arrays solve this by letting us define custom compound data types (`dtypes`), similar to C structs or database rows:

In [6]:
import numpy as np


# Define a custom dtype: string of length 10, 32-bit int, 64-bit float
sensor_dtype = np.dtype([('name', 'U10'), ('id', 'i4'), ('reading', 'f8')])

sensors = np.array([
    ('Sensor_A', 101, 23.5),
    ('Sensor_B', 102, 19.8)
], dtype=sensor_dtype)

print(sensors)
# You can then query named fields like sensors['reading'] or sensors['name'] as standard vectorized arrays.

[('Sensor_A', 101, 23.5) ('Sensor_B', 102, 19.8)]


### Guiding Question:

Suppose we have an array of sensor names `names = np.array(['Sensor_C', 'Sensor_A', 'Sensor_B'])` and an array of their corresponding temperatures `temps = np.array([30.5, 15.2, 22.1])`.

If we use `sorted_indices = np.argsort(temps)` to find the order of temperatures from lowest to highest, how would you use sorted_indices to print both the names and temperatures in sorted order?

In [1]:
import numpy as np

names = np.array(['Sensor_C', 'Sensor_A', 'Sensor_B'])
temps = np.array([30.5, 15.2, 22.1])

sorted_indices = np.argsort(temps)

sorted_names = names[sorted_indices]
sorted_temps = temps[sorted_indices]

for name, temp in zip(sorted_names, sorted_temps):
    print(f"{name}: {temp}°C")

Sensor_A: 15.2°C
Sensor_B: 22.1°C
Sensor_C: 30.5°C


With structured arrays, we can access specific fields by name (like data['temp']), which returns a standard 1D NumPy array that fully supports vectorized operations, masking, and sorting!

### Guiding Question:

If you want to filter data to retrieve only the full structured records (both name and temperature) where the temperature is greater than 20.0°C, how would you write the single-line boolean masking expression on data?

In [ ]:
import numpy as np

# Custom data type definition
sensor_dtype = np.dtype([('name', 'U10'), ('temp', 'f8')])

# Structured dataset
data = np.array([
    ('Sensor_C', 30.5),
    ('Sensor_A', 15.2),
    ('Sensor_B', 22.1)
], dtype=sensor_dtype)

print(f'{data[data['temp'] > 20.0]}')

[('Sensor_C', 30.5) ('Sensor_B', 22.1)]


Executing `data['temp'] > 20.0` extracts the standard 1D float array of temperatures and builds the boolean mask `[True, False, True]`. Passing that back into `data[...]` filters the structured array to return only the matching complete records